# Figure 9: Model Water Masses

## Import Packages

In [1]:
import pandas as pd
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, zoomed_inset_axes
import matplotlib.ticker as mticker
from matplotlib import path
from matplotlib.colors import TwoSlopeNorm
import matplotlib.cm as cm
from matplotlib.patches import Polygon
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import datetime
import math
import matplotlib.patheffects as pe   # at the top of your notebook

import xarray as xr
from netCDF4 import Dataset
from xgcm import Grid
import cmaps
import cmocean
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import glob

import rioxarray

from dask.diagnostics import ProgressBar

from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d

from pyproj import Transformer

import seawater as sw
import gsw

import calendar

import pickle

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/sitecustomize.py:72: UserWarning: The seawater library is deprecated! Please use gsw instead.
  mod = _real_import(name, globals, locals, fromlist, level)


## Import Data

In [2]:
# ---------- OBSERVATIONAL DATA ----------
ROOT = '/g/data/gv90/ad6790/amery-amisor'
os.chdir(f'{ROOT}/notebooks')

from fun_filtering import lowpass_filter

df = pd.read_parquet(f'{ROOT}/data/proc/04_filtered.parquet')

# ---------- NEUTRAL DENSITY CURVES ----------
# Load in the neutral density contours as determined by Blanckensee et al. (2024)
with open("/g/data/gv90/ad6790/amery-amisor/data/external/SiennaNeve-CDObs-203a48c/Notebooks/neu_dens_contours.pkl", "rb") as f: 
    nd_contours = pickle.load(f)
x_curve1 = nd_contours["x_curve1"]  # 28.27 contour
y_curve1 = nd_contours["y_curve1"]
x_curve2 = nd_contours["x_curve2"]  # 28.00 contour
y_curve2 = nd_contours["y_curve2"]

## Define Resuable Functions

In [3]:
def inpolygon(xq, yq, xv, yv):
    shape = xq.shape
    xq = xq.reshape(-1)
    yq = yq.reshape(-1)
    xv = xv.reshape(-1)
    yv = yv.reshape(-1)
    q = [(xq[i], yq[i]) for i in range(xq.shape[0])]
    p = path.Path([(xv[i], yv[i]) for i in range(xv.shape[0])])
    return p.contains_points(q).reshape(shape)

def createMasks(ds):
    ds['mask_zice'] = (ds.mask_rho.where(ds.zice<0)==1) # make a logic mask for ice
    ds['mask_openocean'] = ds.mask_rho*(ds.mask_zice==0)
    ds['mask_openoceanNaN'] = ds.mask_openocean.where(ds.mask_openocean)
    ds['mask_rhoNaN'] = ds.mask_rho.where(ds.mask_rho)
    ds['mask_ziceNaN'] = ds.mask_zice.where(ds.mask_zice)

    poly_shelf = np.array([
        [1.55e6,600000],
        [2.25e6,600000],
        [2.25e6,800000],
        [2.20e6,830000],
        [1.55e6,890000]])
    
    mask_roi = inpolygon(ds.x_rho.values, ds.y_rho.values,poly_shelf[:,0], poly_shelf[:,1])
    ds['mask_zice_roi'] = ds.mask_zice*mask_roi
    roi_label1 = 'amery'
    ds.mask_zice_roi.attrs['long_name']=roi_label1

# Find the coordinates of the four corners of one cell provided
def cell_polygon(j, i, x_psi, y_psi):
    """Corner coords of rho cell (j,i) from its 4 surrounding psi points."""
    # Convert psi (corner) points to a NumPy array
    xp, yp = np.asarray(x_psi), np.asarray(y_psi)
    return [(xp[j-1, i-1], yp[j-1, i-1]),   # SW
            (xp[j-1, i  ], yp[j-1, i  ]),   # SE
            (xp[j,   i  ], yp[j,   i  ]),   # NE
            (xp[j,   i-1], yp[j,   i-1])]   # NW

# Define a function to find the vertical model layer closest to a target depth
def nearest_level(z_col, target_depth):
    """Closest s_rho index to a target depth (both negative, downward)."""
    # Subtract the target from every layer depth, take the absolute value and return the smallest gap
    k = int(np.abs(z_col - target_depth).argmin())
    # Return the level index (k) and the actual depth
    return k, float(z_col[k])

def block_polygon(j, i, x_psi, y_psi, r=1):
    """Outer boundary of a (2r+1)x(2r+1) block of rho cells centred on (j,i)."""
    xp, yp = np.asarray(x_psi), np.asarray(y_psi)
    return [(xp[j-1-r, i-1-r], yp[j-1-r, i-1-r]),   # SW
            (xp[j-1-r, i+r  ], yp[j-1-r, i+r  ]),   # SE
            (xp[j+r,   i+r  ], yp[j+r,   i+r  ]),   # NE
            (xp[j+r,   i-1-r], yp[j+r,   i-1-r])]   # NW
    
def nearest_cell(lon0, lat0, lon_rho, lat_rho):
    lon  = np.deg2rad(np.asarray(lon_rho)); lat  = np.deg2rad(np.asarray(lat_rho))
    lon0r, lat0r = np.deg2rad(lon0), np.deg2rad(lat0)
    dlon, dlat = lon - lon0r, lat - lat0r
    a = np.sin(dlat/2)**2 + np.cos(lat0r)*np.cos(lat)*np.sin(dlon/2)**2
    dist = 2*np.arcsin(np.sqrt(a))                      # great-circle distance
    j, i = np.unravel_index(np.nanargmin(dist), dist.shape)   # flat argmin -> (eta, xi)
    return int(j), int(i)

def style_map(ds, ax, boreholes_amisor, wct_colour=None):
    """Apply the common map overlays and extent to a panel."""
    # ax.contour(ds.x_rho, ds.y_rho, ds.h*ds.mask_rho, levels=(0,250,500,750,1000,2000,3000), colors='grey', linewidths=1)
    ax.contour(ds.x_rho, ds.y_rho, ds.zice*ds.mask_rho, levels=(-1,0), colors='red', linestyles='--', linewidths=0.75)

    wct = (ds.h + ds.zice) * ds.mask_rho # zice is negative, so this is h - |zice|
    if wct_colour is not None:
        cw = ax.contour(ds.x_rho, ds.y_rho, wct, levels=np.arange(0, 3000, 200), colors=wct_colour, linewidths=0.5, zorder=6)
    # ax.clabel(cw, inline=True, fontsize=6, fmt='%.0f')
        
    # Zoomed out
    # ax.set_xlim(1.68e6, 2.45e6)
    # ax.set_ylim(0.52e6, 1.0e6)

    # Eastern flank zoomed in 
    # ax.set_xlim(1.975e6, 2.45e6)
    # ax.set_ylim(0.52e6, 0.75e6)

    # Just AIS
    ax.set_xlim(1.975e6, 2.25e6)
    ax.set_ylim(0.58e6, 0.84e6)

    # Set an equal aspect ratio so that the map doesn't get stretched
    ax.set_aspect('equal', adjustable='box')
    
    ax.xaxis.set_major_formatter(ticks_x)
    ax.yaxis.set_major_formatter(ticks_y)
    for name, info in boreholes_amisor.items():
        lon, lat = info['lonlat']
        bx, by = ll_to_xy.transform(lon, lat)                     # true site -> metres
        ax.plot(bx, by, '.', color='black', ms=10, zorder=8,
                path_effects=[pe.withStroke(linewidth=2, foreground='white')])
        # ax.annotate(name, xy=(bx, by), xytext=(6, 0), textcoords='offset points', color='blue', va='center', zorder=6, bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.4))
        if name in ("AM01", "AM04", "AM05"):
            dx, dy, ha, va = 6, 0, "left", "center"
        elif name == "AM02":
            dx, dy, ha, va = -6, 0, "right", "center"
        elif name == "AM03":
            dx, dy, ha, va = 0, 6, "center", "bottom"
        elif name == "AM06":
            dx, dy, ha, va = 0, -6, "center", "top"
        ax.annotate(name, xy=(bx, by), xytext=(dx, dy), textcoords='offset points', color='black', ha=ha, va=va, zorder=8,
                    path_effects=[pe.withStroke(linewidth=1.5, foreground='white')])

# Define a function to label the neutral density curves tangentially
def label_curve_tangent(ax, x_curve, y_curve, fraction, text, color, fontsize, **kwargs):
    n = len(x_curve)
    # Define an index for the anchor point based on the length of the curve and the fractional distance along it
    idx = int(fraction * (n - 1))
    # Define the anchor point
    anchor_point = (x_curve[idx], y_curve[idx])

    # Pick points neighbouring the anchoring point to estimate the tangent
    i0 = max(idx - 1, 0)
    i1 = min(idx + 1, n - 1)

    # Convert these points to display coordinates so the angle accounts of the axis scaling
    p0 = ax.transData.transform((x_curve[i0], y_curve[i0]))
    p1 = ax.transData.transform((x_curve[i1], y_curve[i1]))
    # Calculate the change in display in display coordinates in the x and y directions
    dx_display = p1[0] - p0[0] 
    dy_display = p1[1] - p0[1] 

    # Calculate the angle of the slope 
    angle = np.degrees(np.arctan2(dy_display, dx_display))
    
    # Find a perpendicular unit vector in display coordinates to create a buffer between the curve and the label 
    length = np.hypot(dx_display, dy_display)
    perp_x = -dy_display / length
    perp_y = dx_display / length

    # Find the anchor point and the offset point from here in display coordinates
    anchor_display = ax.transData.transform(anchor_point)
    offset_display = (anchor_display[0] + 10 * perp_x, anchor_display[1] + 10 * perp_y)

    # Convert this offset point back into data coordinates to be able to plot it correctly
    offset_point = ax.transData.inverted().transform(offset_display)

    # Check if the anchor point falls within the plot's visible area and if it doesn't we ignore this label
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    if not (xlim[0] <= anchor_point[0] <= xlim[1] and ylim[0] <= anchor_point[1] <= ylim[1]):
        return 

    ax.text(offset_point[0], offset_point[1], text, rotation=angle, rotation_mode="anchor", ha="center", va="center", color=color, fontsize=fontsize, **kwargs)

def clip_to_freezing(x, y):
    """Keep only the part of a T-S curve at or above the surface freezing point."""
    d = y - gsw.CT_freezing(x, 0, 0)
    keep = d >= 0
    if not keep.any():
        return x[keep], y[keep]
    i = np.argmax(keep)                       # first point above freezing
    if i == 0:
        return x[keep], y[keep]
    f = -d[i-1] / (d[i] - d[i-1])
    x0 = x[i-1] + f * (x[i] - x[i-1])
    y0 = y[i-1] + f * (y[i] - y[i-1])
    return np.concatenate([[x0], x[keep]]), np.concatenate([[y0], y[keep]])

## Import Model Output

### Load Datasets

In [4]:
# ---------- GRID ----------
grd = xr.open_dataset('/g/data/jk72/deg581/apom/frc/amery_v2km_v13_grd.nc')

# ---------- MODEL OUTPUT ----------
ds_CTRL = xr.open_dataset('/g/data/gv90/ad6790/amery-amisor/data/proc/ctrl_climatology.nc', chunks={})
print(ds_CTRL.nbytes/1e9,'G')

# ---------- LIMA ----------
lima_full = rioxarray.open_rasterio('/g/data/jk72/deg581/apom/ana/data/raw/lima/00000-20080319-092059124.tif')
lima = lima_full.where((lima_full.x>1.68e6)&(lima_full.x<2.5e6)&(lima_full.y>.4e6)&(lima_full.y<1e6),drop=True)

40.109673352 G


### Handle Loaded Datasets

In [5]:
# ---------- GRID ----------
grd['hc']=ds_CTRL.hc
grd['s_w']=ds_CTRL.s_w
grd['s_rho']=ds_CTRL.s_rho
grd['Cs_w']=ds_CTRL.Cs_w
grd['Cs_r']=ds_CTRL.Cs_r

createMasks(grd)

# ---------- MODEL OUTPUT ----------
with ProgressBar():
    ds_CTRL = ds_CTRL.load()

[########################################] | 100% Completed | 499.37 s


### Find AMISOR Borehole Locations

In [6]:
# Initialise a dictionary to hold information on the physical boreholes and moorings
# Borehole lon/lats are at deployment (most change across recording period by less than ~2 km, and maximum ~5 km) 
# Index 0 is coordinates (lon, lat)
# Index 1 is a list of mooring depths in dbar (calculated from average pressure at deployment)
boreholes_amisor = {
    'AM01': {'lonlat': (71.417083, -69.442033), 'avg_pressures': [441.1, 581.6, 743.4], 'ice_draft': 422, 'ocean_floor': 783},
    'AM02': {'lonlat': (72.639983, -69.713767), 'avg_pressures': [338.5, 562.7, 770.2], 'ice_draft': 324, 'ocean_floor': 790},
    'AM03': {'lonlat': (70.332167, -70.561167), 'avg_pressures': [653.6, 861.5, 1211.1], 'ice_draft': 648, 'ocean_floor': 1265},
    'AM04': {'lonlat': (70.290333, -69.8995), 'avg_pressures': [550.4, 693.0, 814.7], 'ice_draft': 539, 'ocean_floor': 938},
    'AM05': {'lonlat': (69.675, -70.231667), 'avg_pressures': [580.5, 738.8, 876.6], 'ice_draft': 554, 'ocean_floor': 909},
    'AM06': {'lonlat': (71.363817, -70.236083), 'avg_pressures': [551.4, 680.8, 790.3], 'ice_draft': 533, 'ocean_floor': 827}
}

# Create a resuable transformation for converting (lon,lat) -> (x,y)
ll_to_xy = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True) 

## Define Water Mass Classification

In [7]:
# To exclude points defined as ice shelf but are not, we mask to a minimum ice draft
MIN_ICE_DRAFT = 100

# AIS cavity mask
cavity_mask = (grd.mask_zice_roi.values > 0) & (-ds_CTRL.zice.values >= MIN_ICE_DRAFT)

# Define a box bounding the cavity in the model so that we can ignore much of the regional ocean
# Find the rows and columns which contain any cavity cell
jrow = np.where(cavity_mask.any(axis=1))[0]
icol = np.where(cavity_mask.any(axis=0))[0]
# Find the first and last row and column containing a cavity cell
J0, J1 = jrow.min(), jrow.max() + 1
I0, I1 = icol.min(), icol.max() + 1

cav_box = cavity_mask[J0:J1, I0:I1]

# Find model layer depths cropped to the cavity box
depth3d = (-ds_CTRL.z_rho0.values[:, J0:J1, I0:I1]).astype(np.float32)
# Find model layer thickness (layers are not all of equal thickness) cropped to the cavity box
dz3d = ds_CTRL.dz.values[:, J0:J1, I0:I1].astype(np.float32)

# Sort Blanckensee's neutral density curves in order of increasing temperature so that we can later use np.interp
# Sort the 28.00 curve as list of positions ordered for temperature to be increasing
sort_order_2800 = np.argsort(y_curve2)
# Put temps and salinities it that order
curve2800_temp = np.asarray(y_curve2)[sort_order_2800]
curve2800_salt = np.asarray(x_curve2)[sort_order_2800]

# Sort the 28.27 curve as list of positions ordered for temperature to be increasing
sort_order_2827 = np.argsort(y_curve1)
# Put temps and salinities it that order
curve2827_temp = np.asarray(y_curve1)[sort_order_2827]
curve2827_salt = np.asarray(x_curve1)[sort_order_2827]

# Define the temp and salinity thresholds for defining HSSW in T-S space
HSSW_TEMP_RANGE = (-1.95, -1.85)
HSSW_SALT_MIN = 34.67

# Define the temp threshold for defining mSW in T-S space
MSW_TEMP_RANGE = (-1.85, -1.7)

# Define a custom colourmap for mSW, replacing the pale yellow start with white on the YlOrBr colourmap
ylorbr = plt.get_cmap('YlOrBr')
msw_cmap = mcolors.LinearSegmentedColormap.from_list('mSW_white', ['white'] + [ylorbr(x) for x in np.linspace(0.2, 1, 8)])

# Define water masses and different colourmaps for each
WATER_MASSES = ['ISW', 'HSSW', 'AASW', 'mCDW', 'mSW']
WM_CMAPS = {'ISW': 'cmo.ice', 'HSSW': 'Purples', 'AASW': 'cmo.tempo', 'mCDW': 'cmo.amp', 'mSW': msw_cmap}

# Define a function to find how much of each water column at each timestep is occupied by each defined water mass
def classify_column_thickness(temp, salt, dz):
    # Initialise a dictionary to hold the thickness of each water mass
    thickness = {}
    for water_mass in WATER_MASSES:
        thickness[water_mass] = np.zeros(temp.shape[1:], dtype=np.float32)

    # Iterate over the water column one level at a time
    for water_column_level in range(temp.shape[0]):
        level_temp = temp[water_column_level]
        level_salt = salt[water_column_level]
        level_thickness = dz[water_column_level]

        # Define the surface freezing point temperature
        surface_freezing = gsw.CT_freezing(level_salt, 0, 0)

        # Find the position in T-S space of this level relative to the neutral density curves
        # Check if fresher than the interpolated neutral density curve at this level's temperature
        left_of_neutral_density_curves = level_salt < np.interp(level_temp, curve2800_temp, curve2800_salt)                                              # gamma_n < 28.00
        # Check if between the interpolated neutral density curves at this level's temperature
        between_neutral_density_curves = (~left_of_neutral_density_curves) & (level_salt < np.interp(level_temp, curve2827_temp, curve2827_salt))        # 28.00 to 28.27
        # >>> NEW: check if saltier than the 28.27 curve at this level's temperature, i.e. denser than 28.27
        right_of_neutral_density_curves = level_salt > np.interp(level_temp, curve2827_temp, curve2827_salt)                                            # gamma_n > 28.27

        # Classify the water at this level in the water column at this point in time (HSSW takes precedence over ISW)
        hssw = (level_temp > HSSW_TEMP_RANGE[0]) & (level_temp < HSSW_TEMP_RANGE[1]) & (level_salt > HSSW_SALT_MIN)
        thickness['HSSW'] += np.where(hssw, level_thickness, 0)
        thickness['ISW']  += np.where((level_temp < surface_freezing) & ~hssw, level_thickness, 0)
        thickness['AASW'] += np.where((level_temp > surface_freezing) & left_of_neutral_density_curves & ~hssw, level_thickness, 0)
        thickness['mCDW'] += np.where((level_temp > surface_freezing) & between_neutral_density_curves & ~hssw, level_thickness, 0)
        thickness['mSW']  += np.where((level_temp > MSW_TEMP_RANGE[0]) & (level_temp < MSW_TEMP_RANGE[1]) & right_of_neutral_density_curves, level_thickness, 0)

    return thickness

# Define a function to take a list of day-of-year values and return, for each water mass, a map of the average thickness of that water mass over that period
def occurrence(doys):
    # Get the index positions of the model's timesteps that fall into this day-of-year range
    index = np.where(np.isin(ds_CTRL.dayofyear.values, doys))[0]
    
    # Initialise a dictionary to hold the running total of thickness of each water mass
    counts = {}
    for water_mass in WATER_MASSES:
        counts[water_mass] = np.zeros(cav_box.shape)

    # Iterate over timestep, t, in this period
    for n, timestep in enumerate(index):
        # Find the temp and salinity at this timestep
        temp = ds_CTRL.temp_doy.values[timestep][:, J0:J1, I0:I1].astype(np.float32)
        salt = ds_CTRL.salt_doy.values[timestep][:, J0:J1, I0:I1].astype(np.float32)
        # Find the thickness of each water mass at the given timestep
        thickness = classify_column_thickness(temp, salt, dz3d)
        # Add this fractional water column breakdown to the running totals
        for water_mass in WATER_MASSES:
            counts[water_mass] += thickness[water_mass]

    # Initialise a dictionary to hold the output maps of water mass thickness across the time period
    output_map = {}
    for water_mass in WATER_MASSES:
        full = np.full(cavity_mask.shape, np.nan)
        # Divide running totals by the number of timesteps to the get the average fraction
        full[J0:J1, I0:I1] = np.where(cav_box, counts[water_mass] / len(index), np.nan)
        output_map[water_mass] = full
    return output_map
 
# OPTIONAL: take a sample of all simulated T-S datapoints as every single datapoint would be tens of millions of points
# Define the how much to thin the data, taking every nth cavity cell (CELL_STEP) and every nth timestep (TIME_STEP)
CELL_STEP = 3
TIME_STEP = 2
 
# Find the row and column of every cavity cell and then keep only every nth cell
ts_jj, ts_ii = np.where(cavity_mask)
ts_jj, ts_ii = ts_jj[::CELL_STEP], ts_ii[::CELL_STEP]
print(f"{cavity_mask.sum()} cavity cells, {ts_jj.size} used for T-S plots")

# Define a function to find the temp, salinity and depth for every sampled cavity cell over a defined period
def cavity_ts(doys):
    # Find which timesteps are in the defined period
    tsel = np.isin(ds_CTRL.dayofyear.values, doys)
    
    # Find the temp, salinity, and depth of each sampled cell over the defined period 
    ct = ds_CTRL.temp_doy.values[tsel][::TIME_STEP][:, :, ts_jj, ts_ii]
    sa = ds_CTRL.salt_doy.values[tsel][::TIME_STEP][:, :, ts_jj, ts_ii]
    dz = np.broadcast_to(depth3d[:, ts_jj - J0, ts_ii - I0], ct.shape)
    
    return ct.ravel(), sa.ravel(), dz.ravel()

14955 cavity cells, 4985 used for T-S plots


## Define Sea-Ice Seasons

In [8]:
# Seasons as lists of dayofyear slices. s4 wraps the year end, so it has two.
SEASONS = {
    'Late Summer (DOY 20-80)':   [slice(20, 80)],
    'Early Sea-ice Season (DOY 80-130)': [slice(80, 130)],
    'Sea-ice Season (DOY 130-330)':       [slice(130, 330)],
    'Early Summer Season (DOY 330-20)':       [slice(330, 366), slice(0, 20)],
}

doy_vals = ds_CTRL['dayofyear'].values

def season_doys(slices):
    """Day-of-year values belonging to a season (handles the year-end wrap)."""
    keep = np.zeros(doy_vals.size, dtype=bool)
    for s in slices:
        keep |= (doy_vals >= s.start) & (doy_vals <= s.stop)
    return doy_vals[keep]

def in_season(doy, slices):
    """Boolean mask for observational day-of-year values falling inside a season."""
    keep = np.zeros(len(doy), dtype=bool)
    for s in slices:
        keep |= (doy >= s.start) & (doy <= s.stop)
    return keep

## Define Constant and Common Variables

In [9]:
# Define a dictionary mapping each MicroCAT to it's pre-calculated average pressure
# Average pressure is the mean pressure on deployment
AVG_PRESSURES = {
    "1969": 441.1,    "1970": 581.6,   "1971": 743.4,
    "1623": 338.5,    "1624": 562.7,   "1174": 770.2,
    "4055": 661.4,    "4054": 872.2,   "3883": 1227.1,
    "1972": 550.4,    "1973": 693.0,   "1974": 814.7,
    "7327": 580.5,    "7326": 738.8,   "7322": 876.6,
    "7325": 551.4,    "7324": 680.8,   "7323": 790.3,
}

# Define a colourmap for the water column of model data
cmap = 'cmo.deep'

# Create a linear mapping of depth from 0 to 1 using the approximated shallowest and deepest levels (in metres) from across all borehole locations in the model 
norm = mcolors.Normalize(vmin=0, vmax=1300)

# Initialise a list of salinity values corresponding to the source water mass at ~-1.9C
meltwater_mixing_lines = [34.67]

# Define the limits of the y-axes for plots of temperature and salinity
ylim_temp = (-2.4, -1.2)
xlim_sal = (34.2, 34.899)

# Define if the composite data will determined using mean or median
average = "mean"

obsv_kw = dict(s=2)
freeze_kw = dict(color="blue", linestyle="-", lw=0.8, label="_", zorder=4)
isopycnal_kw = dict(colors="black", linestyles="-", linewidths=0.6, alpha=0.2, zorder=1)
neutral_kw = dict(color="r", linestyle="-", lw=1.5, zorder=4)
gade_kw = dict(color="k", linestyle="--", lw=1.5, label="_", zorder=5)
label_kw = dict(fontsize=12)
grid_labels=10
# location is edge of plots colourbar is anchored to (options: right, left, top, bottom), shrink scales colourbar's length, pad is gap between coloubar and plots as fraction of axes size, anchor=(X, Y) is X% from left edge and Y% from bottom
# colourbar_kw = dict(orientation='horizontal')
# loc is the point of the legend box pinned to bbox_to_anchor, bbox_to_anchor=(X, Y) is X% from left edge and Y% from bottom, ncol is number of columns in legend entry
legend_kw = dict(ncol=2, fontsize=10, frameon=False, markerscale=2, fancybox=False)
model_kw = dict(cmap=cmap, norm=norm, s=8, edgecolors='none', zorder=2)

water_mass_kwargs = dict(shading='gouraud', rasterized=True)

plt.rcParams.update({
    'font.size': 13,          # base size, everything else scales from it
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
})

## PLOT

In [ ]:
plt.cla()
plt.clf()

scale_ticks = 1e3
ticks_x = mticker.FuncFormatter(lambda x, pos: '{0:g}'.format(x/scale_ticks))
ticks_y = mticker.FuncFormatter(lambda x, pos: '{0:g}'.format(x/scale_ticks))
 
PERIODS = {}
for season_name, season_slices in SEASONS.items():
    PERIODS[season_name] = season_doys(season_slices)
 
# Compute the occurrence percentages once per period
occurences = {}
for period_name, doys in PERIODS.items():
    occurences[period_name] = occurrence(doys)

rows = len(WATER_MASSES) + 1
columns = len(PERIODS)
 
fig, ax = plt.subplots(rows, columns, figsize=(16, 3.3 * rows), layout='constrained', squeeze=False)
# fig, ax = plt.subplots(nrows=5, ncols=4, figsize=(16,17), layout='constrained')

land = np.where(ds_CTRL.mask_rho.values == 0, 1.0, np.nan)
land_cmap = mcolors.ListedColormap(['lightgrey'])

# Define the range of temp and salinity over which to plot overlays
salinity_range = np.linspace(xlim_sal[0], xlim_sal[1], 100)
temp_range = np.linspace(ylim_temp[0], ylim_temp[1], 100)

# Find one colour scale maximum per water mass, taken across all periods
wm_vmax = {}
for water_mass in WATER_MASSES:
    all_periods = np.stack([occurences[period_name][water_mass] for period_name in PERIODS])
    # Round to the 10 m nearest to the 98th percentile thickness 
    wm_vmax[water_mass] = np.ceil(np.nanpercentile(all_periods, 98) / 10) * 10

for row, water_mass in enumerate(WATER_MASSES):
    for col, period_name in enumerate(PERIODS):
        axis = ax[row, col]
        axis.set_facecolor('lightblue')
        axis.pcolormesh(ds_CTRL.x_rho, ds_CTRL.y_rho, land, cmap=land_cmap, shading='auto', zorder=0.5)
 
        # Percentage of timesteps in which this water mass is present anywhere in thecolumn. masked_invalid hides the cells outside the cavity.
        im = axis.pcolormesh(ds_CTRL.x_rho, ds_CTRL.y_rho, np.ma.masked_invalid(occurences[period_name][water_mass]), cmap=WM_CMAPS[water_mass], vmin=0, vmax=wm_vmax[water_mass], **water_mass_kwargs)

        axis.set_title('')
        axis.set_ylabel('')
        axis.set_xlabel('')
        axis.tick_params(axis="both", direction="in", top=True, right=True)
        if row == len(WATER_MASSES) - 1:
            axis.set_xlabel('Eastings (km)')
        else:
            axis.tick_params(labelbottom=False)
        if col == 0:
            axis.set_ylabel('Northings (km)')
        else:
             axis.tick_params(labelleft=False)
            
        style_map(ds_CTRL, axis, boreholes_amisor, wct_colour='lightgrey')

        k = row * columns + col
        axis.text(0.02, 0.98, f'{chr(97 + k)})', transform=axis.transAxes, ha='left', va='top', fontweight='bold', zorder=10)
        # >>> NEW: name the water mass in the lower-left corner of the first panel in each row
        if col == 0:
            axis.text(0.05, 0.05, water_mass, transform=axis.transAxes, ha='left', va='bottom', fontweight='bold', zorder=10)
 
    # One colourbar per row, since each water mass has its own colourmap
    cbar = fig.colorbar(im, ax=ax[row, :], location='right', shrink=0.8, pad=0.02)
    cbar.set_label(f'Avg Thickness (m)')
    cbar.ax.tick_params(labelsize=12)

# ---------- BOTTOM ROW: T-S ----------
ts_row = len(WATER_MASSES)
for col, (pname, doys) in enumerate(PERIODS.items()):
    axis = ax[ts_row, col]
    axis.set_xlim(*xlim_sal)
    axis.set_ylim(*ylim_temp)
 
    # -- PLOT MODEL DATA FROM EVERY (SUBSAMPLED) CAVITY CELL --
    ct, sa, dz = cavity_ts(doys)
    # Shuffle so that no single depth range is systematically drawn on top
    order = np.random.default_rng(0).permutation(ct.size)
    axis.scatter(sa[order], ct[order], c=dz[order], cmap=cmap, norm=norm, s=2, edgecolors='none', zorder=2, rasterized=True)
 
    # -- PLOT OVERLAYS --
    # - Freezing point temperature line at 0 dbar (saturation fraction = 0) -
    CT_freeze_0 = gsw.CT_freezing(salinity_range, 0, 0)
    axis.plot(salinity_range, CT_freeze_0, **freeze_kw)
    label_curve_tangent(axis, salinity_range, CT_freeze_0, 0.125, r"$\Theta_f$ (0 dbar)", "black", grid_labels, zorder=3)

    # - Isopycnals (potential density anomaly contours) referenced to the average pressure of ALL MicroCATs
    # Reference pressure for the potential density anomaly (dbar)
    reference_pressure = np.mean(list(AVG_PRESSURES.values()))
    print(f"Reference Pressure for Plot: {reference_pressure}")
        
    salinity_grid, temp_grid = np.meshgrid(salinity_range, temp_range)
    sigma = gsw.rho(salinity_grid, temp_grid, reference_pressure) - 1000
    cs = axis.contour(salinity_grid, temp_grid, sigma, **isopycnal_kw)
    axis.clabel(cs, inline=True, fontsize=grid_labels, fmt="%.2f")
 
    # - Neutral density curves from Blanckensee et al. (2024) -
    xc1, yc1 = clip_to_freezing(np.asarray(x_curve1), np.asarray(y_curve1))
    xc2, yc2 = clip_to_freezing(np.asarray(x_curve2), np.asarray(y_curve2))
    axis.plot(xc1, yc1, **neutral_kw)
    axis.plot(xc2, yc2, **neutral_kw)
    # label_curve_tangent(axis, xc2, yc2, 0.115, "28.00", "black", grid_labels)
    # label_curve_tangent(axis, xc1, yc1, 0.15, "28.27", "black", grid_labels)

    # # - Meltwater mixing line using gsw (equation 16 from McDougall et al. (2014)) -
    # # First, assume "the temperature of the glacial ice,215 C, measured from a thermistor cable at AM01" as per Herraiz-Borreguero et al. (2013) (also used in Silvano et al. (2017))
    # t_ice = -15.0
    
    # # REVIEW - WHAT PRESSURE DO WE WANT TO PUT HERE?? (note: slope is not particularly sensitive to this pressure)
    # pressure_source = 550

    # for salinity_source in meltwater_mixing_lines:
    #     # Here, we will define the temp of the source water mass as the surface (0 dbar) freezing point temperature at the given source salinities we have chosen
    #     temp_source = float(gsw.CT_freezing(salinity_source, 0, 0))
    #     dSA_dCT = gsw.ice.melting_ice_SA_CT_ratio(salinity_source, temp_source, pressure_source, t_ice)
    #     slope = 1 / dSA_dCT  # dCT/dSA
    #     CT_gade = temp_source + slope * (salinity_range - salinity_source)
    #     axis.plot(salinity_range, CT_gade, **gade_kw)
 
    label_stroke = [pe.withStroke(linewidth=1.5, foreground='white')]
    # - High Salinity Shelf Water -
    axis.text(34.725, -1.89, "HSSW", color="black", va="bottom", ha="left", zorder=4, path_effects=label_stroke, **label_kw)
    # - Low Salinity Shelf Water -
    # axis.text(34.66, -1.89, "LSSW", color="black", va="bottom", ha="right", zorder=4, path_effects=label_stroke, **label_kw)
    # - Antarctic Surface Water -
    axis.text(34.42, -1.8, "AASW", color="black", va="bottom", ha="left", zorder=4, path_effects=label_stroke, **label_kw)
    # - modified Circumpolar Deep Water -
    axis.text(34.6, -1.55, "mCDW", color="black", va="bottom", ha="left", zorder=4, path_effects=label_stroke, **label_kw)
    # - Ice Shelf Water -
    axis.text(34.68, -2.1, "ISW", color="black", va="top", ha="left", zorder=4, path_effects=label_stroke, **label_kw)
    # - modified Shelf Water -
    axis.text(34.655, -1.8, "mSW", color="black", va="bottom", ha="left", zorder=4, path_effects=label_stroke, **label_kw)

    axis.xaxis.set_major_locator(mticker.MultipleLocator(0.2, offset=0.1))
    axis.yaxis.set_major_locator(mticker.MultipleLocator(0.2, offset=0.1))
    
    axis.set_xlabel(r"$\mathrm{S_{A} \ (g \ kg^{-1})}$", **label_kw)
    axis.tick_params(axis="both", direction="in", top=True, right=True)

    k = ts_row * columns + col
    panel_letter = chr(97 + k) if k < 26 else 'a' + chr(97 + k - 26)
    axis.text(0.02, 0.98, f'{panel_letter})', transform=axis.transAxes, ha='left', va='top', fontweight='bold', zorder=10)

    if col != 0:
        axis.tick_params(labelleft=False)

for j in range(columns):
    ax[0, j].annotate(list(SEASONS)[j], xy=(0.5, 1), xycoords='axes fraction', xytext=(0, 6), textcoords='offset points', ha='center', va='bottom', fontweight='bold')

ax[ts_row, 0].set_ylabel(r"$\Theta$ (°C)", **label_kw)

# Depth colourbar for the T-S row
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
cbar = fig.colorbar(sm, ax=ax[ts_row, :], location='right', shrink=0.8, pad=0.02, extend='both')
cbar.set_label('Depth (m)', **label_kw)
# Tick every 100 metres
cbar.ax.yaxis.set_major_locator(mticker.MultipleLocator(250))
# Flip the colourbar so that shallow is at the top and deep at the bottom
cbar.ax.invert_yaxis()
cbar.ax.tick_params(labelsize=12)
 
fig.canvas.draw()
fig.savefig('/g/data/gv90/ad6790/amery-amisor/notebooks/figures/ModelWaterMasses.jpg', format='jpg', dpi=300, bbox_inches='tight')
plt.show()

Reference Pressure for Plot: 693.0388888888888
Reference Pressure for Plot: 693.0388888888888
Reference Pressure for Plot: 693.0388888888888
Reference Pressure for Plot: 693.0388888888888


<Figure size 640x480 with 0 Axes>